# Mutation Potentials Demo

Minimal demo of `DecoderLogProbPotential` and `compose_mutant_distribution` on a short peptide (`AAA`).

In [36]:
import torch
from pep_compass.models.encoder_decoder.hydramp_encoder_decoder import (
    HydrAMPEncoderDecoder,
)
from pep_compass.models.encoder_decoder.utils import decoder_jacobian
from pep_compass.local_enumeration.mutation.utils import get_mutations_from_s_u_standard
from pep_compass.local_enumeration.mutation.mutation_potentials import (
    DecoderLogProbPotential,
    compose_mutant_distribution,
)

In [37]:
device = torch.device("cpu")
hydramp = HydrAMPEncoderDecoder(
    jacobian_mode="approx",
    jacobian_eps=1e-6,
    field_eps=1e-6,
    device=device,
)

In [38]:
peptide = "AFLKFLFKFAA"
pep_len = len(peptide)
z = hydramp.encode_peptides([peptide])
print(f"Parent: {peptide}")
print(f"Latent shape: {z.shape}")

Parent: AFLKFLFKFAA
Latent shape: torch.Size([1, 64])


In [39]:
jac = decoder_jacobian(
    lambda x: hydramp.decoder_forward(x, softmax=True, flatten=True),
    z,
    jacobian_fn_mode="approx",
    jacobian_fn_kwargs={"jacobian_eps": 1e-6},
)
U, S, _ = torch.linalg.svd(jac, full_matrices=False)
print(f"U shape: {U.shape}, S shape: {S.shape}")

U shape: torch.Size([1, 525, 64]), S shape: torch.Size([1, 64])


In [40]:
mutations = get_mutations_from_s_u_standard(
    s=S[0].detach().cpu().numpy(),
    u=U[0].detach().cpu().numpy(),
    max_len=25,
    alphabet_size=21,
    direction_significance_threshold=1e-6,
    min_number_of_directions=5,
    token_threshold=0.05,
)
##shorten mutations to pep_len
mutations = {pos: indices for pos, indices in mutations.items() if pos < pep_len}


alphabet = list(" ACDEFGHIKLMNPQRSTVWY")
print("Proposed mutations:")
for pos in sorted(mutations.keys()):
    aas = [alphabet[i] for i in mutations[pos]]
    print(f"  pos {pos}: indices {mutations[pos]}  ->  {aas}")

Proposed mutations:
  pos 0: indices [6, 16, 6, 16]  ->  ['G', 'S', 'G', 'S']
  pos 1: indices [20]  ->  ['Y']
  pos 2: indices [4, 12, 4, 12]  ->  ['E', 'N', 'E', 'N']
  pos 3: indices [19, 14, 19]  ->  ['W', 'Q', 'W']
  pos 4: indices [13, 20, 8, 13]  ->  ['P', 'Y', 'I', 'P']
  pos 5: indices [12]  ->  ['N']
  pos 6: indices [20, 2, 13, 20]  ->  ['Y', 'C', 'P', 'Y']
  pos 7: indices [12, 13]  ->  ['N', 'P']
  pos 8: indices [13, 20, 8, 13, 8, 13]  ->  ['P', 'Y', 'I', 'P', 'I', 'P']


In [41]:
mutations

{2: [4, 12, 4, 12],
 3: [19, 14, 19],
 0: [6, 16, 6, 16],
 8: [13, 20, 8, 13, 8, 13],
 6: [20, 2, 13, 20],
 7: [12, 13],
 4: [13, 20, 8, 13],
 1: [20],
 5: [12]}

In [42]:
potential = DecoderLogProbPotential(encoder_decoder=hydramp)

df = compose_mutant_distribution(
    parent_peptide=peptide,
    mutations=mutations,
    potential=potential,
    include_parent_residue=True,
    top_k=20,
)

df

MutantDistribution(sequences=['AFLKFLFKFAA', 'AFNKFLFKFAA', 'SFLKFLFKFAA', 'AFLWFLFKFAA', 'AFEKFLFKFAA', 'AFLKFLYKFAA', 'AFLKFLFKYAA', 'GFLKFLFKFAA', 'AFLKFLFNFAA', 'AYLKFLFKFAA', 'AFLKPLFKFAA', 'AFLKFLFPFAA', 'AFLKFLCKFAA', 'AFLKFLFKIAA', 'AFLQFLFKFAA', 'AFLKFNFKFAA', 'AFLKYLFKFAA', 'AFLKFLFKPAA', 'AFLKFLPKFAA', 'AFLKILFKFAA'], log_potentials=array([-9.89435070e-06, -1.21856785e+01, -1.37938499e+01, -1.37947447e+01,
       -1.38917713e+01, -1.49826946e+01, -1.50346622e+01, -1.51009464e+01,
       -1.52184544e+01, -1.52973062e+01, -1.57361758e+01, -1.59718552e+01,
       -1.63978863e+01, -1.65528736e+01, -1.66768067e+01, -1.69409527e+01,
       -1.70159113e+01, -1.70184975e+01, -1.74002151e+01, -1.76605971e+01]))

In [43]:
potentials_dict = potential.compute(peptide, mutations)
print("Raw per-position potentials (first 3 positions):")
for pos in sorted(potentials_dict.keys())[:3]:
    entries = {
        alphabet[aa]: f"{lp:.4f}"
        for aa, lp in sorted(potentials_dict[pos].items(), key=lambda x: -x[1])
    }
    print(f"  pos {pos}: {entries}")

Raw per-position potentials (first 3 positions):
  pos 0: {'S': '-13.7938', 'G': '-15.1009'}
  pos 1: {'Y': '-15.2973'}
  pos 2: {'N': '-12.1857', 'E': '-13.8918'}
